# Circulant Matrix Recovery

This notebook demonstrates recovery of an unknown **circulant** matrix $A(\mathbf{c}^*)$ parameterized by its first row:
$$A(\mathbf{c}) = \sum_{i=0}^{n-1} \mathbf{c}(i)\, C_i$$
where $C_i$ is the canonical circulant matrix with a single 1 in position $(0, i)$ of the first row.

## Circulant Basis

The basis consists of $n$ circulant matrices, each with a single 1 in the first row.
This directly parameterizes all circulant matrices by their first row coefficients.

In [1]:
import numpy as np
from scipy.linalg import circulant
from wias import CPQR_solver

rng = np.random.default_rng(1)

# Matrix dimension = basis size
N_basis = 1000

# Generate canonical circulant basis matrices
# C_k is circulant with first row [0, ..., 0, 1, 0, ..., 0] (1 at position k)
basis_matrices = []
for k in range(N_basis):
    first_row = np.zeros(N_basis)
    first_row[k] = 1.0
    basis_matrices.append(circulant(first_row))

## Generate Unknown Matrix

Create the unknown coefficient vector $\mathbf{c}^* = [c_0^*, c_1^*, \ldots, c_{n-1}^*]$ (the first row)
and construct $A(\mathbf{c}^*) = \sum_{i=0}^{n-1} c_i^* \, C_i$.

In [2]:
# True coefficients (first row of the unknown circulant matrix)
c_true = rng.standard_normal(N_basis)
A_true = sum(c * C_i for c, C_i in zip(c_true, basis_matrices))

## Observe matrix-vector products

We observe:
- **Right** products: $u_j = A(\mathbf{c}^*) f_j$ for $j = 1, \ldots, q_R$
- **Left** products: $v_k = A(\mathbf{c}^*)^\top g_k$ for $k = 1, \ldots, q_L$

In [3]:
# Generate random input vectors
q_R = 1  # number of right inputs
q_L = 0  # number of left inputs

f_vecs = rng.standard_normal((q_R, N_basis))
g_vecs = rng.standard_normal((q_L, N_basis))

# Compute outputs
u_vecs = np.array([A_true @ f for f in f_vecs])
v_vecs = np.array([A_true.T @ g for g in g_vecs])

# Build linear recovery system
q_R, n = f_vecs.shape
q_L, _ = g_vecs.shape

m_total = q_R * n + q_L * n
A_sys = np.zeros((m_total, N_basis))

for i, C_i in enumerate(basis_matrices):
    u_contrib = np.array([C_i @ f for f in f_vecs]).ravel()
    v_contrib = np.array([C_i.T @ g for g in g_vecs]).ravel()
    
    A_sys[:q_R*n, i] = u_contrib
    A_sys[q_R*n:, i] = v_contrib

b_sys = np.concatenate([u_vecs.ravel(), v_vecs.ravel()])

print(f"\nRecovery system : {A_sys.shape}")



Recovery system : (1000, 1000)


## Solve for $\mathbf{c}^*$

In [4]:
c_recovered = CPQR_solver(A_sys, b_sys)

rec_error_direct = np.linalg.norm(c_recovered - c_true)
print("-" * 50)
print(f"L2 recovery error: {rec_error_direct:.2e}")

--------------------------------------------------
L2 recovery error: 1.96e-13
